# Future Sales Prediction - Kaggle Competition

### Modeling File

https://www.kaggle.com/competitions/competitive-data-science-predict-future-sales/

#### File descriptions
- sales_train.csv - the training set. Daily historical data from January 2013 to October 2015.

- test.csv - the test set. You need to forecast the sales for these shops and products for November 2015.

- sample_submission.csv - a sample submission file in the correct format.

- items.csv - supplemental information about the items/products.

- item_categories.csv  - supplemental information about the items categories.

- shops.csv- supplemental information about the shops.

#### Data fields

- ID - an Id that represents a (Shop, Item) tuple within the test set

- shop_id - unique identifier of a shop

- item_id - unique identifier of a product

- item_category_id - unique identifier of item category

- item_cnt_day - number of products sold. You are predicting a monthly amount of this measure

- item_price - current price of an item

- date - date in format dd/mm/yyyy

- date_block_num - a consecutive month number, used for convenience. January 2013 is 0, February 2013 is 1,..., October 2015 is 33

- item_name - name of item

- shop_name - name of shop

- item_category_name - name of item category

### I. Import the dataset and data analysis libraries

In [254]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [255]:
sales_train = pd.read_csv("data/sales_train.csv")
sales_train.head()

,date,date_block_num,shop_id,item_id,item_price,item_cnt_day
0,02.01.2013,0,59,22154,999.00,1.0
1,03.01.2013,0,25,2552,899.00,1.0
2,05.01.2013,0,25,2552,899.00,-1.0
3,06.01.2013,0,25,2554,1709.05,1.0
4,15.01.2013,0,25,2555,1099.00,1.0


In [256]:
test = pd.read_csv("data/test.csv")
test.head()

,ID,shop_id,item_id
0,0,5,5037
1,1,5,5320
2,2,5,5233
3,3,5,5232
4,4,5,5268


In [257]:
sample_submission = pd.read_csv("data/sample_submission.csv")
print(sample_submission.shape)
sample_submission.head()

(214200, 2)


,ID,item_cnt_month
0,0,0.5
1,1,0.5
2,2,0.5
3,3,0.5
4,4,0.5


In [258]:
items = pd.read_csv("data/items.csv")
items.head()

,item_name,item_id,item_category_id
0,! ВО ВЛАСТИ НАВАЖДЕНИЯ (ПЛАСТ.) D,0,40
1,!ABBYY FineReader 12 Professional Edition Full...,1,76
2,***В ЛУЧАХ СЛАВЫ (UNV) D,2,40
3,***ГОЛУБАЯ ВОЛНА (Univ) D,3,40
4,***КОРОБКА (СТЕКЛО) D,4,40


In [259]:
item_categories = pd.read_csv("data/item_categories.csv")
item_categories.head()

,item_category_name,item_category_id
0,PC - Гарнитуры/Наушники,0
1,Аксессуары - PS2,1
2,Аксессуары - PS3,2
3,Аксессуары - PS4,3
4,Аксессуары - PSP,4


In [260]:
shops = pd.read_csv("data/shops.csv")
shops.head()

,shop_name,shop_id
0,"!Якутск Орджоникидзе, 56 фран",0
1,"!Якутск ТЦ ""Центральный"" фран",1
2,"Адыгея ТЦ ""Мега""",2
3,"Балашиха ТРК ""Октябрь-Киномир""",3
4,"Волжский ТЦ ""Волга Молл""",4


### II. Initial data checks and manipulations

In [261]:
sales_train['date'] = pd.to_datetime(sales_train['date'], format = '%d.%m.%Y')
sales_train.head()

,date,date_block_num,shop_id,item_id,item_price,item_cnt_day
0,2013-01-02,0,59,22154,999.00,1.0
1,2013-01-03,0,25,2552,899.00,1.0
2,2013-01-05,0,25,2552,899.00,-1.0
3,2013-01-06,0,25,2554,1709.05,1.0
4,2013-01-15,0,25,2555,1099.00,1.0


In [262]:
# Split into month, day, and year
sales_train['year'] = sales_train['date'].dt.year
sales_train['month'] = sales_train['date'].dt.month
sales_train['day'] = sales_train['date'].dt.day
sales_train.head()

,date,date_block_num,shop_id,item_id,item_price,item_cnt_day,year,month,day
0,2013-01-02,0,59,22154,999.00,1.0,2013,1,2
1,2013-01-03,0,25,2552,899.00,1.0,2013,1,3
2,2013-01-05,0,25,2552,899.00,-1.0,2013,1,5
3,2013-01-06,0,25,2554,1709.05,1.0,2013,1,6
4,2013-01-15,0,25,2555,1099.00,1.0,2013,1,15


In [263]:
sales_train.isna().sum()

date              0
date_block_num    0
shop_id           0
item_id           0
item_price        0
item_cnt_day      0
year              0
month             0
day               0
dtype: int64

In [264]:
sales_train.nunique()

date               1034
date_block_num       34
shop_id              60
item_id           21807
item_price        19993
item_cnt_day        198
year                  3
month                12
day                  31
dtype: int64

In [265]:
# Describing datetime and quantitative variables
sales_train[['date', 'item_price', 'item_cnt_day']].describe()

,date,item_price,item_cnt_day
count,2935849,2.935849e+06,2.935849e+06
mean,2014-04-03 05:44:34.970681344,8.908532e+02,1.242641e+00
min,2013-01-01 00:00:00,-1.000000e+00,-2.200000e+01
25%,2013-08-01 00:00:00,2.490000e+02,1.000000e+00
50%,2014-03-04 00:00:00,3.990000e+02,1.000000e+00
75%,2014-12-05 00:00:00,9.990000e+02,1.000000e+00
max,2015-10-31 00:00:00,3.079800e+05,2.169000e+03
std,NaN,1.729800e+03,2.618834e+00


In [266]:
# Declaring categorical variables
sales_train['shop_id'] = pd.Categorical(sales_train['shop_id'])
sales_train['item_id'] = pd.Categorical(sales_train['item_id'])

In [267]:
# Convert negative to positive values in item_cnt_day
sales_train['item_cnt_day'] = sales_train['item_cnt_day'].abs()

In [268]:
# Create a "master table" for EDA in training data
sales_full = sales_train.merge(items, how = 'inner', on = 'item_id')\
                              .merge(item_categories, how = 'inner', on = 'item_category_id')\
                              .merge(shops, how = 'inner', on = 'shop_id')

sales_full.head()

,date,date_block_num,shop_id,item_id,item_price,item_cnt_day,year,month,day,item_name,item_category_id,item_category_name,shop_name
0,2013-01-02,0,59,22154,999.00,1.0,2013,1,2,ЯВЛЕНИЕ 2012 (BD),37,Кино - Blu-Ray,"Ярославль ТЦ ""Альтаир"""
1,2013-01-03,0,25,2552,899.00,1.0,2013,1,3,DEEP PURPLE The House Of Blue Light LP,58,Музыка - Винил,"Москва ТРК ""Атриум"""
2,2013-01-05,0,25,2552,899.00,1.0,2013,1,5,DEEP PURPLE The House Of Blue Light LP,58,Музыка - Винил,"Москва ТРК ""Атриум"""
3,2013-01-06,0,25,2554,1709.05,1.0,2013,1,6,DEEP PURPLE Who Do You Think We Are LP,58,Музыка - Винил,"Москва ТРК ""Атриум"""
4,2013-01-15,0,25,2555,1099.00,1.0,2013,1,15,DEEP PURPLE 30 Very Best Of 2CD (Фирм.),56,Музыка - CD фирменного производства,"Москва ТРК ""Атриум"""


#### Data checks:

1. Is there any item sold at more than one different price?

In [269]:
shops_and_items = sales_full[['shop_id', 'item_id', 'item_price']].copy()
shops_and_items.drop_duplicates(['shop_id', 'item_id'], inplace = True)

# Same shop, same item, different prices?
print(len(shops_and_items[shops_and_items.duplicated(subset = ['shop_id', 'item_id', 'item_price'])]))

0


2. Is there any group of items having different IDs but the same name? Do the same with shop name and item category name.

In [270]:
print(items.duplicated(['item_id']).sum()) # Number of duplicated item IDs
print(items.duplicated(['item_name']).sum()) # Number of duplicated item names

0
0


In [271]:
# With item categories
print(item_categories.duplicated(['item_category_id']).sum())
print(item_categories.duplicated(['item_category_name']).sum())

0
0


In [272]:
# With shops
print(shops.duplicated(['shop_id']).sum())
print(shops.duplicated(['shop_name']).sum())

0
0


3. Is there any day when the shop can't sell any item?

In [273]:
# Generate list of datetime indices - all days of sales in the training data
datetime_indices = sales_full[['date', 'date_block_num']].copy()
datetime_indices.drop_duplicates(inplace = True)
datetime_indices.sort_values('date', inplace = True)
datetime_indices.set_index('date', inplace = True)
print(datetime_indices.index)

DatetimeIndex(['2013-01-01', '2013-01-02', '2013-01-03', '2013-01-04',
               '2013-01-05', '2013-01-06', '2013-01-07', '2013-01-08',
               '2013-01-09', '2013-01-10',
               ...
               '2015-10-22', '2015-10-23', '2015-10-24', '2015-10-25',
               '2015-10-26', '2015-10-27', '2015-10-28', '2015-10-29',
               '2015-10-30', '2015-10-31'],
              dtype='datetime64[ns]', name='date', length=1034, freq=None)


In [274]:
# Generate a list of dates from the first and last day of sales in the training data
date_range = pd.date_range(start = sales_full['date'].min(), end = sales_full['date'].max())
print(date_range)

DatetimeIndex(['2013-01-01', '2013-01-02', '2013-01-03', '2013-01-04',
               '2013-01-05', '2013-01-06', '2013-01-07', '2013-01-08',
               '2013-01-09', '2013-01-10',
               ...
               '2015-10-22', '2015-10-23', '2015-10-24', '2015-10-25',
               '2015-10-26', '2015-10-27', '2015-10-28', '2015-10-29',
               '2015-10-30', '2015-10-31'],
              dtype='datetime64[ns]', length=1034, freq='D')


### III. Feature engineering

Here, we consider ```date_block_num``` variable as number of months elapsed after January 2013, and this column can also be useful for splitting into different sets.

Aggregate existing features:

In [275]:
# Keep the necessary columns only - we are predicting by month
sales_features = sales_full[['year', 'month', 'date_block_num', 'shop_id', 'item_category_id', 'item_id', 'item_price', 'item_cnt_day']].copy()

# Because no item has more than 1 price, we can group item price by the median
sales_features_price = sales_features[['year', 'month', 'date_block_num', 'shop_id', 'item_category_id', 'item_id', 'item_price']]\
    .groupby(['year', 'month', 'date_block_num', 'shop_id', 'item_category_id', 'item_id'], as_index = False).median()

# Group item count by month by the sum
sales_features_itemcnt = sales_features[['year', 'month', 'date_block_num', 'shop_id', 'item_category_id', 'item_id', 'item_cnt_day']]\
    .groupby(['year', 'month', 'shop_id', 'item_category_id', 'item_id'], as_index = False).sum()

sales_features_itemcnt.rename(columns = {'item_cnt_day': 'item_cnt_month'}, inplace = True)

sales_features = sales_features_price.merge(sales_features_itemcnt, how = 'inner', on = ['year', 'month', 'date_block_num', 'shop_id', 'item_category_id', 'item_id'])
sales_features.sort_values(['year', 'month', 'date_block_num', 'shop_id', 'item_category_id', 'item_id', 'item_price', 'item_cnt_month'], inplace = True)
sales_features.head()

,year,month,date_block_num,shop_id,item_category_id,item_id,item_price,item_cnt_month
0,2013,1,0,0,2,5572,1322.0,10.0
1,2013,1,0,0,2,5573,560.0,1.0
2,2013,1,0,0,2,5575,806.0,4.0
3,2013,1,0,0,2,5576,2231.0,5.0
4,2013,1,0,0,2,5609,2381.0,1.0


Create revenue features (by each (shop_id, item_id) pair):

In [276]:
# Revenue = Price * Item count
sales_features['monthly_item_revenue'] = sales_features['item_price'] * sales_features['item_cnt_month']

sales_features.head()

,year,month,date_block_num,shop_id,item_category_id,item_id,item_price,item_cnt_month,monthly_item_revenue
0,2013,1,0,0,2,5572,1322.0,10.0,13220.0
1,2013,1,0,0,2,5573,560.0,1.0,560.0
2,2013,1,0,0,2,5575,806.0,4.0,3224.0
3,2013,1,0,0,2,5576,2231.0,5.0,11155.0
4,2013,1,0,0,2,5609,2381.0,1.0,2381.0


Create total revenue features (aggregated by shops and item categories):

In [277]:
revenue_shops = sales_features[['year', 'month', 'shop_id', 'monthly_item_revenue']]
revenue_shops = revenue_shops.groupby(['year', 'month', 'shop_id'], as_index = False).sum()
revenue_shops.rename(columns = {'monthly_item_revenue': 'monthly_shop_revenue'}, inplace = True)

revenue_shops.head()

,year,month,shop_id,monthly_shop_revenue
0,2013,1,0,2963911.000
1,2013,1,1,1526256.000
2,2013,1,2,1097686.800
3,2013,1,3,560136.500
4,2013,1,4,1444032.155


In [278]:
revenue_category = sales_features[['year', 'month', 'item_category_id', 'monthly_item_revenue']]
revenue_category = revenue_category.groupby(['year', 'month', 'item_category_id'], as_index = False).sum()
revenue_category.rename(columns = {'monthly_item_revenue': 'monthly_category_revenue'}, inplace = True)

revenue_category.head()

,year,month,item_category_id,monthly_category_revenue
0,2013,1,0,1.480000e+02
1,2013,1,1,1.480000e+02
2,2013,1,2,2.882182e+06
3,2013,1,3,2.105620e+05
4,2013,1,4,2.384025e+05


Create mean revenue per item features (aggregated by shops and item categories):

In [279]:
# Find total number of items sold in a particular shop for each month
total_items_shop = sales_features[['year', 'month', 'shop_id', 'item_cnt_month']]
total_items_shop = total_items_shop.groupby(['year', 'month', 'shop_id'], as_index = False).sum()
total_items_shop.rename(columns = {'item_cnt_month': 'monthly_shop_items'}, inplace = True)

# Merge the total items in a month with the revenue for that particular month to find the mean revenue per item
total_items_shop = total_items_shop.merge(revenue_shops, on = ['year', 'month', 'shop_id'])
total_items_shop['mean_shop_rev_per_item'] = total_items_shop['monthly_shop_revenue'] / total_items_shop['monthly_shop_items']
total_items_shop.head()

,year,month,shop_id,monthly_shop_items,monthly_shop_revenue,mean_shop_rev_per_item
0,2013,1,0,5578.0,2963911.000,531.357297
1,2013,1,1,2947.0,1526256.000,517.901595
2,2013,1,2,1156.0,1097686.800,949.556055
3,2013,1,3,767.0,560136.500,730.295306
4,2013,1,4,2120.0,1444032.155,681.147243


In [280]:
# Do the same process with item categories
total_items_category = sales_features[['year', 'month', 'item_category_id', 'item_cnt_month']]
total_items_category = total_items_category.groupby(['year', 'month', 'item_category_id'], as_index = False).sum()
total_items_category.rename(columns = {'item_cnt_month': 'monthly_category_items'}, inplace = True)

total_items_category = total_items_category.merge(revenue_category, on = ['year', 'month', 'item_category_id'])
total_items_category['mean_cat_rev_per_item'] = total_items_category['monthly_category_revenue'] / total_items_category['monthly_category_items']
total_items_category.head()

,year,month,item_category_id,monthly_category_items,monthly_category_revenue,mean_cat_rev_per_item
0,2013,1,0,1.0,1.480000e+02,148.000000
1,2013,1,1,1.0,1.480000e+02,148.000000
2,2013,1,2,1400.0,2.882182e+06,2058.701585
3,2013,1,3,442.0,2.105620e+05,476.384615
4,2013,1,4,259.0,2.384025e+05,920.472973


Create weekend and holiday features:

In [281]:
# Separate the sales days together
sales_days = sales_full[['date']].copy()
sales_days.drop_duplicates(inplace = True, ignore_index = True)
sales_days.sort_values('date', inplace = True, ignore_index = True)

# Create month, year, and weekend indicator variables
sales_days['year'] = sales_days['date'].dt.year
sales_days['month'] = sales_days['date'].dt.month
sales_days['weekend_indicator'] = sales_days['date'].case_when(
    [(sales_days['date'].dt.day_of_week.isin([5, 6]), 1),
     (~sales_days['date'].dt.day_of_week.isin([5, 6]), 0)]
)

sales_days.head(7)

,date,year,month,weekend_indicator
0,2013-01-01,2013,1,0
1,2013-01-02,2013,1,0
2,2013-01-03,2013,1,0
3,2013-01-04,2013,1,0
4,2013-01-05,2013,1,1
5,2013-01-06,2013,1,1
6,2013-01-07,2013,1,0


In [282]:
# See Analysis File for more details
# List of holidays in Russia: https://www.timeanddate.com/holidays/russia/2013 (same as for 2014 and 2015)
holiday_list_2013 = ['2013-01-01', '2013-01-02', '2013-01-03', '2013-01-04', '2013-01-05', '2013-01-06', '2013-01-07', '2013-01-08',
                     '2013-02-23', '2013-03-08', '2013-05-01', '2013-05-02', '2013-05-03', '2013-05-09', '2013-05-10',
                     '2013-06-12', '2013-11-04']

holiday_list_2014 = ['2014-01-01', '2014-01-02', '2014-01-03', '2014-01-06', '2014-01-07', '2014-01-08',
                     '2014-02-22', '2014-02-23', '2014-03-08', '2014-03-09', '2014-03-10', '2014-05-01', '2014-05-02',
                     '2014-05-03', '2014-05-09', '2014-05-10', '2014-05-11', '2014-06-12', '2014-06-13', '2014-06-14', '2014-06-15',
                     '2014-11-01', '2014-11-02', '2014-11-03', '2014-11-04']

holiday_list_2015 = ['2015-01-01', '2015-01-02', '2015-01-03', '2015-01-04', '2015-01-05', '2015-01-06', '2015-01-07',
                     '2015-01-08', '2015-01-09', '2015-02-23', '2015-03-08', '2015-03-09', '2015-05-01', '2015-05-04', '2015-05-09',
                     '2015-05-11', '2015-06-12'] # only count until October 2015

# Combine to have a list of holidays in training data
holidays = holiday_list_2013 + holiday_list_2014 + holiday_list_2015
holidays = [pd.Timestamp(x) for x in holidays] # Convert the whole list to timestamp format

# Create holiday indicator variable
sales_days['holiday_indicator'] = sales_days['date'].case_when(
    [(sales_days['date'].isin(holidays), 1),
     (~sales_days['date'].isin(holidays), 0)]
)

sales_days.head()

,date,year,month,weekend_indicator,holiday_indicator
0,2013-01-01,2013,1,0,1
1,2013-01-02,2013,1,0,1
2,2013-01-03,2013,1,0,1
3,2013-01-04,2013,1,0,1
4,2013-01-05,2013,1,1,1


In [283]:
# Count the number of days as weekends and holidays in a (month, year) pair
sales_days.drop('date', axis = 1, inplace = True)
sales_days = sales_days.groupby(['year', 'month'], as_index = False).sum()
sales_days.rename(columns = {'weekend_indicator': 'weekends_in_month', 'holiday_indicator': 'holidays_in_month'}, inplace = True)
sales_days.head(7)

,year,month,weekends_in_month,holidays_in_month
0,2013,1,8,8
1,2013,2,8,1
2,2013,3,10,1
3,2013,4,8,0
4,2013,5,8,5
5,2013,6,10,1
6,2013,7,8,0


Create moving and lagged metrics (all shops and items, aggregated monthly):

In [284]:
sales_month = sales_features[['year', 'month', 'item_cnt_month']].copy()
sales_month = sales_month.groupby(['year', 'month'], as_index = False).sum()

# Lag features for 1 month and 3 months
sales_month['lag_1m'] = sales_month['item_cnt_month'].shift(periods = 1)
sales_month['lag_3m'] = sales_month['item_cnt_month'].shift(periods = 3)

# Percentage (fractional) change by 1 month and 3 months
sales_month['pct_change_1m'] = sales_month['item_cnt_month'].pct_change(periods = 1)
sales_month['pct_change_3m'] = sales_month['item_cnt_month'].pct_change(periods = 3)

# Moving average sales for the last 3 months
sales_month['moving_3m'] = sales_month['item_cnt_month'].rolling(window = 3).mean()

# Imputation by backfill for missing data
# Assume that sales data by month doesn't change before Jan 2013
sales_month.bfill(inplace = True)

# Rename a column further clarification
sales_month.rename(columns = {'item_cnt_month': 'total_monthly_quantity'}, inplace = True)

sales_month.head(7)

,year,month,total_monthly_quantity,lag_1m,lag_3m,pct_change_1m,pct_change_3m,moving_3m
0,2013,1,132221.0,132221.0,132221.0,-0.685784,-0.706801,72329.666667
1,2013,2,41546.0,132221.0,132221.0,-0.685784,-0.706801,72329.666667
2,2013,3,43222.0,41546.0,132221.0,0.040341,-0.706801,72329.666667
3,2013,4,38767.0,43222.0,132221.0,-0.103073,-0.706801,41178.333333
4,2013,5,37629.0,38767.0,41546.0,-0.029355,-0.094281,39872.666667
5,2013,6,39154.0,37629.0,43222.0,0.040527,-0.094119,38516.666667
6,2013,7,40510.0,39154.0,38767.0,0.034632,0.044961,39097.666667


Create seasonal features:

In [285]:
sales_month['season'] = sales_month['month'].case_when(
    [(sales_month['month'].isin([3, 4, 5]), 'spring'),
     (sales_month['month'].isin([6, 7, 8]), 'summer'),
     (sales_month['month'].isin([9, 10, 11]), 'fall'),
     (sales_month['month'].isin([12, 1, 2]), 'winter')]
)

# Because winter is very cold in Russia, I will encode so that winter goes first
sales_month['season'] = pd.Categorical(values = sales_month['season'], ordered = True,
                                       categories = ['winter', 'spring', 'summer', 'fall'])

# Use one-hot encoding to turn the seasons into features
# Drop the first column (winter) to avoid multicollinearity
onehot_season = pd.get_dummies(sales_month['season'], drop_first = True)

# Convert data type of all the one-hot encoded values: True/False to 0/1
onehot_season = onehot_season.map(lambda x: int(x))

# Merge the one-hot encoding into the sales_month feature DataFrame
sales_month = pd.concat([sales_month, onehot_season], axis = 1)

# Drop the categorical 'season' column now that we have 3 one-hot encoded season columns
sales_month.drop('season', axis = 1, inplace = True)

sales_month.head(7)

,year,month,total_monthly_quantity,lag_1m,lag_3m,pct_change_1m,pct_change_3m,moving_3m,spring,summer,fall
0,2013,1,132221.0,132221.0,132221.0,-0.685784,-0.706801,72329.666667,0,0,0
1,2013,2,41546.0,132221.0,132221.0,-0.685784,-0.706801,72329.666667,0,0,0
2,2013,3,43222.0,41546.0,132221.0,0.040341,-0.706801,72329.666667,1,0,0
3,2013,4,38767.0,43222.0,132221.0,-0.103073,-0.706801,41178.333333,1,0,0
4,2013,5,37629.0,38767.0,41546.0,-0.029355,-0.094281,39872.666667,1,0,0
5,2013,6,39154.0,37629.0,43222.0,0.040527,-0.094119,38516.666667,0,1,0
6,2013,7,40510.0,39154.0,38767.0,0.034632,0.044961,39097.666667,0,1,0


Now combine the features together:

In [286]:
final_features = sales_features\
    .merge(total_items_shop, how = 'left', on = ['year', 'month', 'shop_id'])\
    .merge(total_items_category, how = 'left', on = ['year', 'month', 'item_category_id'])\
    .merge(sales_days, how = 'left', on = ['year', 'month'])\
    .merge(sales_month, how = 'left', on = ['year', 'month'])

final_features.head(7)

,year,month,date_block_num,shop_id,item_category_id,item_id,item_price,item_cnt_month,monthly_item_revenue,monthly_shop_items,monthly_shop_revenue,mean_shop_rev_per_item,monthly_category_items,monthly_category_revenue,mean_cat_rev_per_item,weekends_in_month,holidays_in_month,total_monthly_quantity,lag_1m,lag_3m,pct_change_1m,pct_change_3m,moving_3m,spring,summer,fall
0,2013,1,0,0,2,5572,1322.0,10.0,13220.0,5578.0,2963911.0,531.357297,1400.0,2.882182e+06,2058.701585,8,8,132221.0,132221.0,132221.0,-0.685784,-0.706801,72329.666667,0,0,0
1,2013,1,0,0,2,5573,560.0,1.0,560.0,5578.0,2963911.0,531.357297,1400.0,2.882182e+06,2058.701585,8,8,132221.0,132221.0,132221.0,-0.685784,-0.706801,72329.666667,0,0,0
2,2013,1,0,0,2,5575,806.0,4.0,3224.0,5578.0,2963911.0,531.357297,1400.0,2.882182e+06,2058.701585,8,8,132221.0,132221.0,132221.0,-0.685784,-0.706801,72329.666667,0,0,0
3,2013,1,0,0,2,5576,2231.0,5.0,11155.0,5578.0,2963911.0,531.357297,1400.0,2.882182e+06,2058.701585,8,8,132221.0,132221.0,132221.0,-0.685784,-0.706801,72329.666667,0,0,0
4,2013,1,0,0,2,5609,2381.0,1.0,2381.0,5578.0,2963911.0,531.357297,1400.0,2.882182e+06,2058.701585,8,8,132221.0,132221.0,132221.0,-0.685784,-0.706801,72329.666667,0,0,0
5,2013,1,0,0,2,5612,3623.0,1.0,3623.0,5578.0,2963911.0,531.357297,1400.0,2.882182e+06,2058.701585,8,8,132221.0,132221.0,132221.0,-0.685784,-0.706801,72329.666667,0,0,0
6,2013,1,0,0,2,5623,294.0,1.0,294.0,5578.0,2963911.0,531.357297,1400.0,2.882182e+06,2058.701585,8,8,132221.0,132221.0,132221.0,-0.685784,-0.706801,72329.666667,0,0,0


The last preprocessing before I start to train my models:

In [287]:
# Encode month using trigonometric encoding for cyclical features
final_features['month_sin'] = np.sin(final_features['month'] / 12 * 2 * np.pi)
final_features['month_cos'] = np.cos(final_features['month'] / 12 * 2 * np.pi)

# Drop the month variable now that it isn't necessary
final_features.drop('month', axis = 1, inplace = True)

final_features.head(7)

,year,date_block_num,shop_id,item_category_id,item_id,item_price,item_cnt_month,monthly_item_revenue,monthly_shop_items,monthly_shop_revenue,mean_shop_rev_per_item,monthly_category_items,monthly_category_revenue,mean_cat_rev_per_item,weekends_in_month,holidays_in_month,total_monthly_quantity,lag_1m,lag_3m,pct_change_1m,pct_change_3m,moving_3m,spring,summer,fall,month_sin,month_cos
0,2013,0,0,2,5572,1322.0,10.0,13220.0,5578.0,2963911.0,531.357297,1400.0,2.882182e+06,2058.701585,8,8,132221.0,132221.0,132221.0,-0.685784,-0.706801,72329.666667,0,0,0,0.5,0.866025
1,2013,0,0,2,5573,560.0,1.0,560.0,5578.0,2963911.0,531.357297,1400.0,2.882182e+06,2058.701585,8,8,132221.0,132221.0,132221.0,-0.685784,-0.706801,72329.666667,0,0,0,0.5,0.866025
2,2013,0,0,2,5575,806.0,4.0,3224.0,5578.0,2963911.0,531.357297,1400.0,2.882182e+06,2058.701585,8,8,132221.0,132221.0,132221.0,-0.685784,-0.706801,72329.666667,0,0,0,0.5,0.866025
3,2013,0,0,2,5576,2231.0,5.0,11155.0,5578.0,2963911.0,531.357297,1400.0,2.882182e+06,2058.701585,8,8,132221.0,132221.0,132221.0,-0.685784,-0.706801,72329.666667,0,0,0,0.5,0.866025
4,2013,0,0,2,5609,2381.0,1.0,2381.0,5578.0,2963911.0,531.357297,1400.0,2.882182e+06,2058.701585,8,8,132221.0,132221.0,132221.0,-0.685784,-0.706801,72329.666667,0,0,0,0.5,0.866025
5,2013,0,0,2,5612,3623.0,1.0,3623.0,5578.0,2963911.0,531.357297,1400.0,2.882182e+06,2058.701585,8,8,132221.0,132221.0,132221.0,-0.685784,-0.706801,72329.666667,0,0,0,0.5,0.866025
6,2013,0,0,2,5623,294.0,1.0,294.0,5578.0,2963911.0,531.357297,1400.0,2.882182e+06,2058.701585,8,8,132221.0,132221.0,132221.0,-0.685784,-0.706801,72329.666667,0,0,0,0.5,0.866025


#### Explanation of new feature names:

- ```monthly_item_revenue```: Total revenue for a particular item in the month

- ```monthly_shop_items```: Number of items sold in a particular shop in the month (counting all items)

- ```monthly_shop_revenue```: Total revenue for a particular shop in the month

- ```mean_shop_rev_per_item```: Revenue per item of a shop in the month

- ```monthly_category_items```: Number of items of a particular category sold in the month (counting all shops)

- ```mean_cat_rev_per_item```: Revenue per item of an item category in the month

- ```weekends_in_month```: Number of days of weekend (Saturday or Sunday) in the month

- ```holidays_in_month```: Number of days of public holiday in the month

- ```total_monthly_quantity```: Total number of items sold in the month (counting all shops and all items aggregated)

- ```lag_1m```, ```lag_3m```: 1-month/ 3-month lagged feature of ```total_monthly_quantity``` (backfilled for the first months)

- ```pct_change_1m```, ```pct_change_3m```: Fractional change of ```total_monthly_quantity``` for each month (backfilled for the first months)

- ```moving_3m```: Moving average of ```total_monthly_quantity``` for the 3 latest months (backfilled for the first months)

- ```spring```, ```summer```, ```fall```: Dummy variables denoting the season that an item was sold. If ```spring = summer = fall = 0```, then this item was sold in winter.

### IV. Declare training and validation sets

We are using 5-fold time series split cross-validation. However, because there are more than 1 observation in a single time frame (panel data), I have to split the folds manually.

Splitting plan (using date_block_num): There are 34 months in the training data (indexed from 0 to 33). I will split the folds so that for the smallest fold, there is at least sufficient data for a year to capture the cyclical characteristics of time series.

| Fold | Fold's training data          | Fold's validation data   |
|------|-------------------------------|--------------------------|
| 1    | ```date_block_num```: 0 -> 11 | ```date_block_num```: 12 |
| 2    | ```date_block_num```: 0 -> 17 | ```date_block_num```: 18 |
| 3    | ```date_block_num```: 0 -> 23 | ```date_block_num```: 24 |
| 4    | ```date_block_num```: 0 -> 29 | ```date_block_num```: 30 |
| 5    | ```date_block_num```: 0 -> 32 | ```date_block_num```: 33 |

In [288]:
# Split into the corresponding folds

# Fold 1
fold_1_train = final_features[final_features['date_block_num'].isin(range(12))]
fold_1_test = final_features[final_features['date_block_num'] == 12]

# Fold 2
fold_2_train = final_features[final_features['date_block_num'].isin(range(18))]
fold_2_test = final_features[final_features['date_block_num'] == 18]

# Fold 3
fold_3_train = final_features[final_features['date_block_num'].isin(range(24))]
fold_3_test = final_features[final_features['date_block_num'] == 24]

# Fold 4
fold_4_train = final_features[final_features['date_block_num'].isin(range(30))]
fold_4_test = final_features[final_features['date_block_num'] == 30]

# Fold 5
fold_5_train = final_features[final_features['date_block_num'].isin(range(33))]
fold_5_test = final_features[final_features['date_block_num'] == 33]

In [289]:
# Number of data points in each fold
print(f'''Fold 1: {fold_1_train.shape[0]} items in training set, {fold_1_test.shape[0]} items in validation set.
      Validation / Training = {fold_1_test.shape[0] / fold_1_train.shape[0]}''')
print('-------------------------------------------------------------')
print(f'''Fold 2: {fold_2_train.shape[0]} items in training set, {fold_2_test.shape[0]} items in validation set.
      Validation / Training = {fold_2_test.shape[0] / fold_2_train.shape[0]}''')
print('-------------------------------------------------------------')
print(f'''Fold 3: {fold_3_train.shape[0]} items in training set, {fold_3_test.shape[0]} items in validation set.
      Validation / Training = {fold_3_test.shape[0] / fold_3_train.shape[0]}''')
print('-------------------------------------------------------------')
print(f'''Fold 4: {fold_4_train.shape[0]} items in training set, {fold_4_test.shape[0]} items in validation set.
      Validation / Training = {fold_4_test.shape[0] / fold_4_train.shape[0]}''')
print('-------------------------------------------------------------')
print(f'''Fold 5: {fold_5_train.shape[0]} items in training set, {fold_5_test.shape[0]} items in validation set.
      Validation / Training = {fold_5_test.shape[0] / fold_5_train.shape[0]}''')

Fold 1: 482291 items in training set, 35056 items in validation set.
      Validation / Training = 0.07268640716911574
-------------------------------------------------------------
Fold 2: 674998 items in training set, 31685 items in validation set.
      Validation / Training = 0.04694087982482911
-------------------------------------------------------------
Fold 3: 857397 items in training set, 29839 items in validation set.
      Validation / Training = 0.03480184791875875
-------------------------------------------------------------
Fold 4: 1008819 items in training set, 23354 items in validation set.
      Validation / Training = 0.023149841547393538
-------------------------------------------------------------
Fold 5: 1075730 items in training set, 21820 items in validation set.
      Validation / Training = 0.020283900235189126


In [290]:
# Split into features and targets

# Fold 1
X_train_1 = fold_1_train.drop('item_cnt_month', axis = 1).values
y_train_1 = fold_1_train[['item_cnt_month']].values.ravel()
X_test_1 = fold_1_test.drop('item_cnt_month', axis = 1).values
y_test_1 = fold_1_test[['item_cnt_month']].values.ravel()

# Fold 2
X_train_2 = fold_2_train.drop('item_cnt_month', axis = 1).values
y_train_2 = fold_2_train[['item_cnt_month']].values.ravel()
X_test_2 = fold_2_test.drop('item_cnt_month', axis = 1).values
y_test_2 = fold_2_test[['item_cnt_month']].values.ravel()

# Fold 3
X_train_3 = fold_3_train.drop('item_cnt_month', axis = 1).values
y_train_3 = fold_3_train[['item_cnt_month']].values.ravel()
X_test_3 = fold_3_test.drop('item_cnt_month', axis = 1).values
y_test_3 = fold_3_test[['item_cnt_month']].values.ravel()

# Fold 4
X_train_4 = fold_4_train.drop('item_cnt_month', axis = 1).values
y_train_4 = fold_4_train[['item_cnt_month']].values.ravel()
X_test_4 = fold_4_test.drop('item_cnt_month', axis = 1).values
y_test_4 = fold_4_test[['item_cnt_month']].values.ravel()

# Fold 5
X_train_5 = fold_5_train.drop('item_cnt_month', axis = 1).values
y_train_5 = fold_5_train[['item_cnt_month']].values.ravel()
X_test_5 = fold_5_test.drop('item_cnt_month', axis = 1).values
y_test_5 = fold_5_test[['item_cnt_month']].values.ravel()

In [291]:
# Check the shape of each X and y in training and test sets
print(f'''Fold 1: X_train: {X_train_1.shape}, y_train: {y_train_1.shape}, X_test: {X_test_1.shape}, y_test: {y_test_1.shape}''')
print(f'''Fold 2: X_train: {X_train_2.shape}, y_train: {y_train_2.shape}, X_test: {X_test_2.shape}, y_test: {y_test_2.shape}''')
print(f'''Fold 3: X_train: {X_train_3.shape}, y_train: {y_train_3.shape}, X_test: {X_test_3.shape}, y_test: {y_test_3.shape}''')
print(f'''Fold 4: X_train: {X_train_4.shape}, y_train: {y_train_4.shape}, X_test: {X_test_4.shape}, y_test: {y_test_4.shape}''')
print(f'''Fold 5: X_train: {X_train_5.shape}, y_train: {y_train_5.shape}, X_test: {X_test_5.shape}, y_test: {y_test_5.shape}''')

Fold 1: X_train: (482291, 26), y_train: (482291,), X_test: (35056, 26), y_test: (35056,)
Fold 2: X_train: (674998, 26), y_train: (674998,), X_test: (31685, 26), y_test: (31685,)
Fold 3: X_train: (857397, 26), y_train: (857397,), X_test: (29839, 26), y_test: (29839,)
Fold 4: X_train: (1008819, 26), y_train: (1008819,), X_test: (23354, 26), y_test: (23354,)
Fold 5: X_train: (1075730, 26), y_train: (1075730,), X_test: (21820, 26), y_test: (21820,)


In [292]:
# Example on fold 1
fold_1_train.tail()

,year,date_block_num,shop_id,item_category_id,item_id,item_price,item_cnt_month,monthly_item_revenue,monthly_shop_items,monthly_shop_revenue,mean_shop_rev_per_item,monthly_category_items,monthly_category_revenue,mean_cat_rev_per_item,weekends_in_month,holidays_in_month,total_monthly_quantity,lag_1m,lag_3m,pct_change_1m,pct_change_3m,moving_3m,spring,summer,fall,month_sin,month_cos
482286,2013,11,59,75,3160,1590.0,1.0,1590.0,685.0,429674.9,627.262628,427.0,1090907.2,2554.817799,9,0,41574.0,35356.0,35355.0,0.175868,0.175902,37443.333333,0,0,0,-2.449294e-16,1.0
482287,2013,11,59,75,4186,1491.0,1.0,1491.0,685.0,429674.9,627.262628,427.0,1090907.2,2554.817799,9,0,41574.0,35356.0,35355.0,0.175868,0.175902,37443.333333,0,0,0,-2.449294e-16,1.0
482288,2013,11,59,75,4688,1499.0,1.0,1499.0,685.0,429674.9,627.262628,427.0,1090907.2,2554.817799,9,0,41574.0,35356.0,35355.0,0.175868,0.175902,37443.333333,0,0,0,-2.449294e-16,1.0
482289,2013,11,59,75,12727,1490.0,1.0,1490.0,685.0,429674.9,627.262628,427.0,1090907.2,2554.817799,9,0,41574.0,35356.0,35355.0,0.175868,0.175902,37443.333333,0,0,0,-2.449294e-16,1.0
482290,2013,11,59,83,22091,109.0,1.0,109.0,685.0,429674.9,627.262628,12.0,986.0,82.166667,9,0,41574.0,35356.0,35355.0,0.175868,0.175902,37443.333333,0,0,0,-2.449294e-16,1.0


In [293]:
# Set seed for reproducibility
my_seed = 275225

In [294]:
# What features can I apply scaling on (with MinMaxScaler)?

features_to_remove = ['shop_id', 'item_category_id', 'item_id', 'month_sin', 'month_cos', 'spring', 'summer', 'fall']
features_list = final_features.columns.tolist().copy() # start with all features that I have included

# Need to use an if statement so that this code can be rerun many times (this includes inplace removal of list elements)
for feature in features_to_remove:
    if feature in features_list:
        features_list.remove(feature)

print(features_list)

['year', 'date_block_num', 'item_price', 'item_cnt_month', 'monthly_item_revenue', 'monthly_shop_items', 'monthly_shop_revenue', 'mean_shop_rev_per_item', 'monthly_category_items', 'monthly_category_revenue', 'mean_cat_rev_per_item', 'weekends_in_month', 'holidays_in_month', 'total_monthly_quantity', 'lag_1m', 'lag_3m', 'pct_change_1m', 'pct_change_3m', 'moving_3m']


In [295]:
# https://stackoverflow.com/questions/22934609/get-indexes-of-multiple-pandas-columns-by-names
# Get index of the columns to pass to ColumnTransformer without error
# https://stackoverflow.com/questions/71715754/valueerror-specifying-the-columns-using-strings-is-only-supported-for-pandas-da

categorical_col_indices = [final_features.columns.get_loc(col) for col in ['shop_id', 'item_category_id', 'item_id']]
print(f'Indices of categorical columns: {categorical_col_indices}')

minmax_indices = [final_features.columns.get_loc(col) for col in features_list]
print(f'Indices of columns to apply MinMaxScaler on: {minmax_indices}')

Indices of categorical columns: [2, 3, 4]
Indices of columns to apply MinMaxScaler on: [0, 1, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21]


In [296]:
from sklearn.preprocessing import TargetEncoder, MinMaxScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Encode categorical variables (shop_id, item_category_id, item_id) with TargetEncoder
# Encode all other variables (except for "month" and dummy variables) with MinMaxScaler, default range [0, 1]

target_enc = TargetEncoder(target_type = 'continuous', shuffle = True, random_state = my_seed)

transformer = ColumnTransformer(
    transformers = [('target_enc', target_enc, categorical_col_indices),
                    ('minmax_scaler', MinMaxScaler(), minmax_indices)],
    remainder = 'passthrough'
)

### V. Implementing regression models and assessing feature importance

In [297]:
from sklearn.metrics import root_mean_squared_error
import statistics
import time # I want to measure the efficiency of different machine learning models

# Custom function for training in many folds and many different models

folds = [[X_train_1, X_test_1, y_train_1, y_test_1], [X_train_2, X_test_2, y_train_2, y_test_2],
         [X_train_3, X_test_3, y_train_3, y_test_3], [X_train_4, X_test_4, y_train_4, y_test_4],
         [X_train_5, X_test_5, y_train_5, y_test_5]]

def kfold_regression(model_name, model, folds):
    rmse_folds = []

    pipe = Pipeline([('transformer', transformer), ('regression', model)])

    # I want to measure the time that Python fits the data and output the scores for all 5 folds
    start_time = time.perf_counter()

    # Each element in "folds" list should have the form [X_train, X_test, y_train, y_test]
    for fold in folds:
        pipe.fit(fold[0], fold[2])
        y_pred = pipe.predict(fold[1])
        rmse_folds.append(root_mean_squared_error(fold[3], y_pred))
    
    end_time = time.perf_counter()

    # Output as a neatly formatted DataFrame
    df = pd.DataFrame({'model_name': [model_name], 'mean_rmse': [statistics.mean(rmse_folds)],
                       'std_rmse': [statistics.stdev(rmse_folds)], 'execution_time': [end_time - start_time]})
    return df

In [298]:
# Import the necessary models
from sklearn.linear_model import LinearRegression, Lasso, Ridge, ElasticNet, SGDRegressor

In [244]:
model_list = [('Linear Regression', LinearRegression(n_jobs = -1), folds),
               ('Lasso', Lasso(random_state = my_seed), folds),
               ('Ridge', Ridge(random_state = my_seed), folds),
               ('Elastic Net', ElasticNet(random_state = my_seed), folds)]

overall_results = pd.concat([kfold_regression(model_name, model, folds) for model_name, model, folds in model_list], axis = 0, ignore_index = True)
overall_results.sort_values(['mean_rmse', 'std_rmse', 'execution_time'], inplace = True, ignore_index = True)
overall_results

,model_name,mean_rmse,std_rmse,execution_time
0,Elastic Net,1.995931,2.520029,18.463380
1,Lasso,1.995931,2.520029,52.281478
2,Linear Regression,2.004312,2.507050,47.729682
3,Ridge,2.013181,2.499405,38.102378


### VI. Hyperparameter tuning (coming soon)

### VII. Prepare data for final submission

In [305]:
test_shop_items = test[['shop_id', 'item_id']].copy()
test_shop_items.head() # this is the test data stripped of its ID

,shop_id,item_id
0,5,5037
1,5,5320
2,5,5233
3,5,5232
4,5,5268


In [306]:
# Set constant data for some columns
test_features = test_shop_items.copy()

test_features['year'] = 2013
test_features['date_block_num'] = 34 # number of months elapsed since Jan 2013

test_features['spring'] = 0; test_features['summer'] = 0; test_features['fall'] = 1 # November is in the fall season
test_features['weekends_in_month'] = 9
test_features['holidays_in_month'] = 1 # Russia's Unity Day in 4 November

# For the month_sin and month_cos trigonometric encoding, replace the month number with 11
test_features['month_sin'] = np.sin(11 / 12 * 2 * np.pi)
test_features['month_cos'] = np.cos(11 / 12 * 2 * np.pi)

test_features.head()

,shop_id,item_id,year,date_block_num,spring,summer,fall,weekends_in_month,holidays_in_month,month_sin,month_cos
0,5,5037,2013,34,0,0,1,9,1,-0.5,0.866025
1,5,5320,2013,34,0,0,1,9,1,-0.5,0.866025
2,5,5233,2013,34,0,0,1,9,1,-0.5,0.866025
3,5,5232,2013,34,0,0,1,9,1,-0.5,0.866025
4,5,5268,2013,34,0,0,1,9,1,-0.5,0.866025


In [307]:
# Retrieve item category and item price (each item only belongs to 1 category and has 1 price)
item_info = final_features[['item_id', 'item_category_id', 'item_price']].copy()

# Merge item information to the test set
item_info = test_shop_items.merge(item_info, on = 'item_id', how = 'left')
item_info.head()

,shop_id,item_id,item_category_id,item_price
0,5,5037,19.0,2599.0
1,5,5037,19.0,2599.0
2,5,5037,19.0,2599.0
3,5,5037,19.0,2599.0
4,5,5037,19.0,2599.0


Remember that there are some items not in the test set. I will fill the null values later.

Remaining columns to address:

monthly_item_revenue, monthly_shop_items, monthly_shop_revenue, mean_shop_rev_per_item, monthly_category_items, monthly_category_revenue, mean_cat_rev_per_item, total_monthly_quantity, lag_1m, lag_3m, pct_change_1m, pct_change_3m, moving_3m